# Merging the Airbnb tables with pandas

The right `merge` for every pair of Airbnb tables: which key, and why.

| Left table | Right table | Key | Why |
|---|---|---|---|
| `calendar` | `listings` | `listing_id` = `id` | both name the same listing; the calendar has one row per night |
| `reviews_monthly` | `listings` | `listing_id` = `id` | both name the same listing; one row per month with reviews |
| `reviews_monthly` | `weather_daily` | `month` | both describe a month in Berlin; the weather is daily, so group it by month first |
| `listings` | `neighbourhoods` | `neighbourhood` | the name of the area on the map |

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Load the tables

In [ ]:
import json

import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")                # one row per listing
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet")                # one row per listing and night
reviews_monthly = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")  # one row per listing and month
weather_daily = pd.read_parquet("case-study/data/airbnb/weather_daily.parquet")      # one row per day

with open("case-study/data/airbnb/neighbourhoods.geojson") as file:
    areas_geojson = json.load(file)
neighbourhoods = pd.DataFrame([feature["properties"] for feature in areas_geojson["features"]])  # one row per area

## The merge syntax

```python
pd.merge(left, right, left_on="key_in_left", right_on="key_in_right", how="left")
pd.merge(left, right, on="key", how="left")   # when the key has the same name in both tables
```

`left` is the first table, `right` the second. `how="left"` keeps every row of the left table; `how="inner"` (the default) keeps only rows whose key is in both tables.

## Calendar and listings

Key: `listing_id` in `calendar` = `id` in `listings`. Each night gets the district and room type of its listing.

In [ ]:
calendar_with_listing = pd.merge(
    calendar,
    listings[["id", "district", "room_type"]],
    left_on="listing_id", right_on="id",
)
calendar_with_listing.head()

## Reviews and listings

Key: `listing_id` in `reviews_monthly` = `id` in `listings`. Each month of reviews gets the district of its listing.

In [ ]:
reviews_with_listing = pd.merge(
    reviews_monthly,
    listings[["id", "district"]],
    left_on="listing_id", right_on="id",
)
reviews_with_listing.head()

## Listings and reviews: one row per listing

To add reviews to `listings` and keep one row per listing, sum the reviews per listing first, then merge. `how="left"` keeps listings without reviews; `fillna(0)` gives them 0.

In [ ]:
reviews_per_listing = reviews_monthly.groupby("listing_id", as_index=False)["n_reviews"].sum()

listings_with_reviews = pd.merge(
    listings[["id", "district"]],
    reviews_per_listing,
    left_on="id", right_on="listing_id",
    how="left",
)
listings_with_reviews["n_reviews"] = listings_with_reviews["n_reviews"].fillna(0)
listings_with_reviews.head()

## Reviews and weather

Key: `month`. The weather has one row per day, so first group it by month; `.dt.to_period("M").dt.to_timestamp()` writes each date as the first day of its month, as in `reviews_monthly`. Then sum the reviews of all listings per month.

In [ ]:
weather_monthly = (
    weather_daily
    .assign(month=weather_daily["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(mean_temperature=("temperature_2m_mean", "mean"))
)
reviews_per_month = reviews_monthly.groupby("month", as_index=False)["n_reviews"].sum()

reviews_and_weather = pd.merge(reviews_per_month, weather_monthly, on="month")
reviews_and_weather.head()

## Listings and neighbourhoods

Key: `neighbourhood`, the same name in both tables, so `on=` is enough. Each listing gets the district from the map (`neighbourhood_group`).

In [ ]:
listings_with_area = pd.merge(
    listings[["id", "neighbourhood"]],
    neighbourhoods,
    on="neighbourhood",
    how="left",
)
listings_with_area.head()

## Further reading

- pandas user guide: [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html).
- The files and their links: [case-study/data/airbnb/PROVENANCE.md](../../../case-study/data/airbnb/PROVENANCE.md).